# CICIDS2017 Phase 1 Exploratory Data Analysis

This notebook documents a read-only EDA of the raw CICIDS2017 MachineLearningCSV files under `data/raw/`. It reports data-quality issues without cleaning or changing the raw dataset.

## Guardrails

- Raw DVC-tracked files are immutable inputs.
- Original multiclass labels are analyzed before final binary target mapping.
- Scans are streaming/chunk-friendly and avoid immediate full concatenation.
- Split discussion is temporal/day/file-based; naive random row-level splits are excluded.

In [ ]:
# Set to True to regenerate reports/eda/summary.json and summary.md from raw CSVs.
# The full scan is read-only but can take several minutes on a local machine.
import runpy
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RUN_FULL_SCAN = False
if RUN_FULL_SCAN:
    runpy.run_path(str(ROOT / 'scripts' / 'phase1_eda.py'), run_name='__main__')

In [ ]:
import json
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SUMMARY_JSON = ROOT / 'reports' / 'eda' / 'summary.json'
with SUMMARY_JSON.open('r', encoding='utf-8') as fh:
    summary = json.load(fh)
print(summary['scan_method'])
print(f"files={summary['overall']['file_count']} rows={summary['overall']['row_count']:,}")

## Discovered CSV Files, Sizes, and Row Counts

In [ ]:
for f in summary['files']:
    print(f"{f['source_filename']}: {f['size_mb']:.3f} MiB, {f['row_count']:,} rows")

## Exact Original Column Names and Whitespace

In [ ]:
for c in summary['original_columns']:
    ws = ' whitespace' if c['leading_whitespace'] or c['trailing_whitespace'] else ''
    print(f"{c['index']:02d}: {c['name']!r} -> {c['trimmed_name']!r}{ws}")
print('duplicate original:', summary['duplicate_original_column_names'])
print('duplicate trimmed:', summary['duplicate_trimmed_column_names'])

## Column Consistency Across Files

In [ ]:
print(json.dumps(summary['column_consistency'], indent=2))

## Data Types Per Feature

In [ ]:
for c in summary['overall']['columns']:
    print(f"{c['index']:02d} {c['name']!r}: {c['dtype_inferred']} {c['dtype_evidence_counts']}")

## Labels Present and BENIGN/Attack Distribution

In [ ]:
print('overall labels:', json.dumps(summary['overall']['labels'], indent=2))
print('overall binary grouping:', summary['overall']['binary_class_distribution'])
for f in summary['files']:
    print(f"{f['source_filename']}: labels={f['labels']} binary={f['binary_class_distribution']}")

## Missing, Infinite, and Duplicate Values

In [ ]:
for c in summary['overall']['columns']:
    if c['missing_count'] or c['positive_infinite_count'] or c['negative_infinite_count']:
        print(c['index'], repr(c['name']), 'missing', c['missing_count'], '+inf', c['positive_infinite_count'], '-inf', c['negative_infinite_count'])
print('overall duplicate rows:', summary['overall']['duplicate_rows'])

## Constant, Near-Constant, Leakage-Prone Columns, and Splits

In [ ]:
print('constant:', summary['overall']['constant_features'])
print('near constant:', summary['overall']['near_constant_features'])
print('suspicious/review:', json.dumps(summary['interpretation']['suspicious_identifier_or_leakage_prone_columns'], indent=2))
print('attack types by file:', json.dumps(summary['interpretation']['attack_types_by_file'], indent=2))
print('memory:', json.dumps(summary['interpretation']['memory_requirements'], indent=2))

## Narrative Report

The narrative conclusions, required cleaning operations, label coverage trade-offs, and proposed split requiring review are in `reports/eda/summary.md`.